# MAICON 공통 스켈레톤

**사용법:** `0. 설정` 셀만 문제에 맞게 고치고 위에서부터 실행한다.
흐름: 규칙 정리 → 로드·점검 → 분할 → **기준선 제출 먼저 저장** → 후보 짝 비교 → 전체 재학습 → 제출 형식 점검.

- 지원: 이진·다중분류, 다중라벨(미검토 라벨 = 빈칸 허용), 회귀 / 특징: 정형·텍스트·긴 형식 시계열(구간 통계)
- 탐지(AP50)·격자(mIoU 전용 특징)는 문제 전용 코드로 작성하고, 지표·짝 비교·제출 점검만 여기 것을 쓴다.

## 규칙 정리 (지문에서 옮겨 적기)

| 항목 | 내용 |
|---|---|
| 지표 | |
| 제출 파일 경로·형식 | |
| 허용/금지 (모델, 가중치, 외부 데이터) | |
| 시간 | |
| 데이터 구조 → 분할 방식 | |

In [ ]:
# ===== 0. 설정: 문제마다 이 셀만 고친다 =====
from pathlib import Path
BASE_DIR    = Path('.')                # 문제 폴더
TRAIN_FILE  = 'train.csv'              # 학습 표 (ID + 정답)
TEST_FILE   = 'test.csv'               # test ID가 있는 표 (ID만 있으면 sample_submission.csv를 써도 됨)
SAMPLE_FILE = 'sample_submission.csv'
ID_COL      = 'id'
TARGET      = 'target'                 # 다중라벨이면 ['a', 'b', ...]
TASK        = 'binary'                 # 'binary' | 'multiclass' | 'multilabel' | 'regression'
METRIC      = 'f1_pos'                 # 'f1_pos' | 'macro_f1' | 'multilabel_macro_f1' | 'miou' | 'accuracy' | 'mae' | 'rmse'
FEATURES    = 'tabular'                # 'tabular' | 'text' | 'series'
TEXT_COL    = None                     # FEATURES='text'일 때 본문 컬럼
SERIES_TRAIN, SERIES_TEST = None, None # FEATURES='series'일 때 긴 형식 시계열 파일 (ID_COL로 묶고, 파일의 행 순서를 시간 순서로 본다)
SPLIT       = 'stratified'             # 'stratified' | 'group' | 'time'
GROUP_COL   = None                     # SPLIT='group'일 때 그룹 컬럼 (입력 특징에서는 빠짐)
N_SPLITS, N_REPEATS = 5, 5             # stratified: 반복마다 OOF를 모아 채점 → 짝 단위 = 반복
ML_THRESHOLD = 0.5                     # 다중라벨 판정 임계값
SEED        = 42
OUT_FILE    = 'submission.csv'
TIME_LIMIT_MIN = 30

In [ ]:
# ===== 공통 import / 타이머 =====
import time, warnings
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold, GroupKFold, TimeSeriesSplit, TunedThresholdClassifierCV
from sklearn.metrics import f1_score, jaccard_score, accuracy_score, mean_absolute_error, mean_squared_error
warnings.filterwarnings('ignore', category=UserWarning)
T0 = time.time()
def remaining():
    used = (time.time() - T0) / 60
    print(f'[시간] 경과 {used:.1f}분 / 남은 {TIME_LIMIT_MIN - used:.1f}분')

## 1. 로드·점검

In [ ]:
read = lambda f: pd.read_csv(BASE_DIR / f, encoding='utf-8-sig')
train, test, sample = read(TRAIN_FILE), read(TEST_FILE), read(SAMPLE_FILE)
TARGETS = TARGET if isinstance(TARGET, list) else [TARGET]

for name, df in [('train', train), ('test', test)]:
    print(f'[{name}] {df.shape}  결측 {int(df.isna().sum().sum())}  중복 ID {int(df[ID_COL].duplicated().sum())}')
print('test ID 순서 = sample 순서:', test[ID_COL].tolist() == sample[ID_COL].tolist())
print('train/test ID 겹침:', len(set(train[ID_COL]) & set(test[ID_COL])))
if TASK == 'regression':
    print(train[TARGET].describe())
elif TASK == 'multilabel':
    print(pd.DataFrame({'양성비율': train[TARGETS].mean(), '미검토(빈칸)': train[TARGETS].isna().sum()}))
else:
    print(train[TARGET].value_counts().sort_index().to_frame('개수').assign(비율=lambda d: (d['개수'] / d['개수'].sum()).round(3)))
remaining()

## 2. 특징 (FEATURES에 따라 자동 선택)

In [ ]:
def series_features(path):
    # 긴 형식 시계열 → ID별 통계: 평균·표준편차·최소·최대·(마지막-처음)
    s = read(path)
    num = s.drop(columns=[ID_COL]).select_dtypes('number')
    num = num.drop(columns=[c for c in ['step', 'time', 't', 'timestamp'] if c in num])
    g = num.groupby(s[ID_COL], sort=False)
    f = pd.concat({'mean': g.mean(), 'std': g.std(), 'min': g.min(), 'max': g.max(), 'delta': g.last() - g.first()}, axis=1)
    f.columns = [f'{b}_{a}' for a, b in f.columns]
    return f

if FEATURES == 'series':
    X      = series_features(SERIES_TRAIN).loc[train[ID_COL]].reset_index(drop=True)
    X_test = series_features(SERIES_TEST).loc[test[ID_COL]].reset_index(drop=True)
elif FEATURES == 'text':
    X, X_test = train[[TEXT_COL]].fillna(''), test[[TEXT_COL]].fillna('')
else:
    drop = [ID_COL] + TARGETS + ([GROUP_COL] if GROUP_COL else [])
    X = train.drop(columns=drop)
    X_test = test[X.columns]

# 분류 라벨은 0..K-1로 인코딩하고 제출 때 되돌린다 (이진은 큰 값이 양성 = 인코딩 1)
if TASK == 'multilabel':
    y = train[TARGETS].to_numpy(dtype=float)        # 빈칸(NaN) = 미검토 → 학습·채점에서 제외
    CLASSES = np.array([0, 1])
elif TASK == 'regression':
    y = train[TARGET].to_numpy(dtype=float)
    CLASSES = None
else:
    CLASSES, y = np.unique(train[TARGET].to_numpy(), return_inverse=True)
groups = train[GROUP_COL].to_numpy() if GROUP_COL else None
print('X', X.shape, 'X_test', X_test.shape, '| 클래스', CLASSES)

## 3. 지표 (채점처럼 OOF 전체를 모아 한 번 계산)

In [ ]:
def multilabel_macro_f1(yt, yp):
    yt, yp = np.asarray(yt, np.float64), np.asarray(yp)
    out = []
    for j in range(yt.shape[1]):
        m = ~np.isnan(yt[:, j])
        out.append(f1_score(yt[m, j].astype(int), yp[m, j].astype(int), zero_division=0))
    return float(np.mean(out))

LABELS = list(range(len(CLASSES))) if CLASSES is not None else None
METRICS = {   # (함수, 방향: +1 높을수록 좋음 / -1 낮을수록 좋음)
    'f1_pos':   (lambda t, p: f1_score(t, p, pos_label=1, zero_division=0), +1),
    'macro_f1': (lambda t, p: f1_score(t, p, labels=LABELS, average='macro', zero_division=0), +1),
    'multilabel_macro_f1': (multilabel_macro_f1, +1),
    'miou':     (lambda t, p: jaccard_score(t, p, labels=LABELS, average='macro', zero_division=0), +1),
    'accuracy': (accuracy_score, +1),
    'mae':      (mean_absolute_error, -1),
    'rmse':     (lambda t, p: float(np.sqrt(mean_squared_error(t, p))), -1),
}
score_fn, DIRECTION = METRICS[METRIC]
print('지표:', METRIC, '(높을수록 좋음)' if DIRECTION > 0 else '(낮을수록 좋음)')

## 4. 분할 (한 번 만들고 모든 후보가 같은 분할을 쓴다)

In [ ]:
def make_splits():
    # 반환: [(짝 단위 번호, train_idx, valid_idx)], 짝 단위 종류
    n = len(X)
    if SPLIT == 'stratified':
        if TASK in ('binary', 'multiclass'):
            cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
            it = cv.split(np.zeros(n), y)
        else:
            it = RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED).split(np.zeros(n))
        return [(i // N_SPLITS, tr, va) for i, (tr, va) in enumerate(it)], 'rep'
    if SPLIT == 'group':
        it = GroupKFold(n_splits=N_SPLITS).split(np.zeros(n), groups=groups)
    elif SPLIT == 'time':   # 행이 시간순이어야 함. 앞부분으로 학습, 뒷부분으로 검증
        it = TimeSeriesSplit(n_splits=N_SPLITS).split(np.zeros(n))
    else:
        raise ValueError(SPLIT)
    return [(i, tr, va) for i, (tr, va) in enumerate(it)], 'fold'

SPLITS, UNIT = make_splits()
print(f'분할 {len(SPLITS)}개, 짝 단위 = {UNIT} ({len(set(u for u, _, _ in SPLITS))}개)')

## 5. 모델 후보 (Pipeline 안에서 전처리 → fold 안에서만 학습되어 누수 방지)

In [ ]:
REG = TASK == 'regression'

def preprocessor(scale=False):
    if FEATURES == 'text':
        return ColumnTransformer([
            ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), TEXT_COL),
            ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True), TEXT_COL)])
    num = X.select_dtypes('number').columns.tolist()
    cat = [c for c in X.columns if c not in num]
    num_steps = [('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])
    return ColumnTransformer([('num', Pipeline(num_steps), num),
                              ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])

def linear():
    m = Ridge() if REG else LogisticRegression(max_iter=3000, C=1.0)
    return Pipeline([('pre', preprocessor(scale=True)), ('m', m)])

def linear_balanced():
    return Pipeline([('pre', preprocessor(scale=True)), ('m', LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced'))])

def hgb():
    m = HistGradientBoostingRegressor(random_state=SEED) if REG else HistGradientBoostingClassifier(random_state=SEED)
    return Pipeline([('pre', preprocessor()), ('m', m)])

BASELINE = 'linear'
CANDIDATES = {'linear': linear}           # 후보와 순서는 결과를 보기 전에 정한다 (CLAUDE.md §5)
if not REG:
    CANDIDATES['linear_balanced'] = linear_balanced
if FEATURES != 'text':                    # 희소 TF-IDF에는 HGB를 쓰지 않는다
    CANDIDATES['hgb'] = hgb
if TASK == 'binary':                      # 임계값 조정: 학습 fold 안의 내부 CV로만 고른다
    CANDIDATES['linear_tuned'] = lambda: TunedThresholdClassifierCV(
        linear(), scoring='f1' if METRIC == 'f1_pos' else 'f1_macro', cv=3, random_state=SEED)
print('후보:', list(CANDIDATES), '| 기준:', BASELINE)

## 6. CV 실행 · 짝 비교 · 제출 도구

In [ ]:
def fit_predict(make_model, Xtr, ytr, Xva):
    if TASK == 'multilabel':   # 라벨별 학습, 미검토(NaN) 행은 그 라벨 학습에서만 제외
        out = np.zeros((len(Xva), ytr.shape[1]))
        for j in range(ytr.shape[1]):
            m = ~np.isnan(ytr[:, j]); yy = ytr[m, j].astype(int)
            if len(np.unique(yy)) < 2:
                out[:, j] = yy[0] if len(yy) else 0; continue
            out[:, j] = make_model().fit(Xtr.iloc[np.flatnonzero(m)], yy).predict_proba(Xva)[:, 1]
        return (out >= ML_THRESHOLD).astype(int)
    return make_model().fit(Xtr, ytr).predict(Xva)

def pooled(oof):
    m = ~np.isnan(oof) if oof.ndim == 1 else ~np.isnan(oof).any(axis=1)
    p = oof[m] if REG else oof[m].astype(int)
    return float(score_fn(y[m], p))

def run_cv(name):
    t = time.time(); oofs, fold_scores = {}, []
    for u, tr, va in SPLITS:
        p = fit_predict(CANDIDATES[name], X.iloc[tr], y[tr], X.iloc[va])
        key = u if UNIT == 'rep' else 0
        oofs.setdefault(key, np.full(y.shape, np.nan))[va] = p
        fold_scores.append(float(score_fn(y[va], p)))
    rep_scores = [pooled(o) for o in oofs.values()]
    unit_scores = np.array(rep_scores if UNIT == 'rep' else fold_scores, np.float64)
    r = dict(name=name, unit_scores=unit_scores, pooled=float(np.mean(rep_scores)), oof=oofs[0])
    print(f'{name:16s} pooled {r["pooled"]:.4f} | 짝 단위 점수 {np.round(unit_scores, 4)} | {time.time() - t:.1f}초')
    return r

def paired_compare(base, cand, tol=1e-9):
    # CLAUDE.md §5: d = 후보가 나을수록 양수, |d| <= 1e-9 동점. 채택 = 평균 d > 0 그리고 개선 짝 > 악화 짝
    d = DIRECTION * (cand['unit_scores'] - base['unit_scores'])
    w, l, t = int((d > tol).sum()), int((d < -tol).sum()), int((np.abs(d) <= tol).sum())
    adopt = bool(d.mean() > 0 and w > l)
    print(f'{cand["name"]:16s} vs {base["name"]}: 짝 {len(d)} | 개선 {w} / 악화 {l} / 동점 {t} | 평균 d {d.mean():+.6f} → {"채택 후보" if adopt else "기준 유지"}')
    if len(d) < 5: print('   (짝 5개 미만: 판정 보류 수준)')
    if t > len(d) / 2: print('   (동점이 대부분: 기대 이득 작음)')
    return adopt, float(d.mean())

def validate_submission(sub):
    assert list(sub.columns) == list(sample.columns), f'컬럼 불일치 {list(sub.columns)} vs {list(sample.columns)}'
    assert len(sub) == len(sample), '행 수 불일치'
    assert sub[ID_COL].tolist() == sample[ID_COL].tolist(), 'ID 순서 불일치'
    assert not sub.isna().any().any(), '결측 있음'
    if not REG:
        allowed = set(CLASSES.tolist())
        for c in TARGETS:
            assert sub[c].isin(allowed).all(), f'{c}: 허용되지 않은 값'
    print(f'제출 형식 OK: {sub.shape}')

def fit_full_and_save(name, path=OUT_FILE):
    p = fit_predict(CANDIDATES[name], X, y, X_test)
    pred = pd.DataFrame(p, columns=TARGETS) if TASK == 'multilabel' else pd.DataFrame({TARGET: p if REG else CLASSES[p]})
    pred.insert(0, ID_COL, test[ID_COL].to_numpy())
    sub = sample[[ID_COL]].merge(pred, on=ID_COL, how='left')[sample.columns]   # ID로 연결
    validate_submission(sub)
    sub.to_csv(BASE_DIR / path, index=False)
    print(f'저장: {BASE_DIR / path} ({name})')
    if not REG:
        print('예측 분포(진단용):', {c: sub[c].value_counts().sort_index().to_dict() for c in TARGETS})
    return sub

## 7. 기준선 → 제출 파일 먼저 저장

In [ ]:
RESULTS = {BASELINE: run_cv(BASELINE)}
fit_full_and_save(BASELINE)
remaining()

## 8. 후보 비교 (같은 분할, 같은 기준선 대비 짝 비교)

In [ ]:
for name in CANDIDATES:
    if name not in RESULTS:
        RESULTS[name] = run_cv(name)
    remaining()

passed = []
for name, r in RESULTS.items():
    if name == BASELINE: continue
    adopt, mean_d = paired_compare(RESULTS[BASELINE], r)
    if adopt: passed.append((mean_d, name))

# 여러 후보가 통과하면 평균 d가 가장 큰 후보 (1e-9 이내 동점이면 후보 순서가 앞선 것)
best = BASELINE
if passed:
    top = max(d for d, _ in passed)
    best = [n for d, n in passed if top - d <= 1e-9][0]
print('선택:', best, '(선택에 쓴 점수이므로 독립적인 성능 확증이 아님)')

## 9. 최종: 전체 학습 → 제출 형식 점검 → 저장

In [ ]:
if best != BASELINE:
    fit_full_and_save(best)
else:
    print('기준선 제출 유지:', BASE_DIR / OUT_FILE)
remaining()